# LOGISTICS-05 — Otimização de Hiperparâmetros de Machine Learning

## 1. Objetivo de negócio

Identificar antecipadamente coletas com maior risco de atraso, permitindo que uma Torre de Controle Logística priorize alertas e intervenções operacionais.

O projeto utiliza o dataset **LaDe-P Shanghai (Pickup)**, com registros históricos de operações de coleta.

O evento de interesse é a ocorrência de coleta após o limite da janela operacional. Não estamos prevendo atrasos de entrega.

## 2. Contexto dos experimentos anteriores

Os notebooks anteriores desenvolveram e avaliaram classificadores utilizando diferentes conjuntos de features.

O notebook `04_comparacao_features_aoi_courier.ipynb` comparou oito experimentos envolvendo Random Forest e XGBoost.

Resultados selecionados:

| Modelo | Features | Average Precision |
|---|---|---:|
| Random Forest D | Baseline + Courier | 0,226512 |
| XGBoost C | Baseline + AOI + Courier | 0,269095 |
| XGBoost D | Baseline + Courier | 0,268009 |

O XGBoost C apresentou a maior Average Precision observada.

Entretanto, o XGBoost D apresentou desempenho próximo, utilizando um conjunto de features mais simples.

Por isso, a primeira rodada de otimização utilizará Random Forest D e XGBoost D.

O XGBoost C permanecerá como referência para comparação e possível otimização posterior.

## 3. Objetivo técnico

Investigar se ajustes nos hiperparâmetros dos classificadores podem melhorar a capacidade de identificar coletas atrasadas, mantendo controle sobre:

- Generalização dos modelos.
- Risco de overfitting.
- Custo computacional.
- Complexidade dos modelos.
- Reprodutibilidade dos experimentos.
- Capacidade operacional de priorização de alertas.

A otimização não garante melhoria de desempenho.

## 4. Metodologia

Será utilizado `RandomizedSearchCV`, do scikit-learn.

A ferramenta seleciona aleatoriamente combinações de hiperparâmetros dentro de espaços de busca previamente definidos.

Cada combinação será avaliada por validação cruzada.

### 4.1 Configuração aprovada

- Random Forest D: 12 iterações.
- XGBoost D: 12 iterações.
- Validação cruzada: 3 folds estratificados.
- Métrica principal: Average Precision.
- Semente de aleatoriedade: 42.
- Total previsto: 72 ajustes de validação cruzada, além de dois reajustes finais caso `refit=True`.

### 4.2 Controle de data leakage

A otimização utilizará exclusivamente o conjunto de treinamento.

O pré-processamento será integrado ao `Pipeline`, garantindo que os parâmetros aprendidos durante a transformação dos dados sejam ajustados dentro de cada fold.

O conjunto de teste reservado de 15% permanecerá isolado até a avaliação final.

A validação de 15% utilizada nos experimentos anteriores não participará da busca de hiperparâmetros.

### 4.3 Avaliação dos resultados

Serão considerados:

- Average Precision média entre os folds.
- Desvio-padrão da Average Precision.
- Diferenças entre desempenho de treinamento e validação cruzada.
- Tempo de treinamento.
- Melhores combinações de hiperparâmetros.
- Recall@K e Precision@K na avaliação operacional posterior.

A seleção final considerará desempenho e custo computacional, não apenas a maior pontuação obtida.

## 5. Reprodutibilidade

Os experimentos anteriores estão preservados no Git.

Commit de referência: `6968e93`.

O ambiente inicial utiliza Python 3.13.16, scikit-learn 1.9.1 e XGBoost 3.4.1.

Os hiperparâmetros, resultados, tempos de execução e decisões metodológicas serão documentados neste notebook.

## 6. Organização do notebook

O desenvolvimento seguirá quatro componentes para cada etapa:

1. Markdown anterior ao código, apresentando objetivo e metodologia.
2. Código Python completo e comentado.
3. Markdown explicativo dos conceitos e da implementação.
4. Markdown posterior à execução, com resultados, interpretação e conclusão.

Nenhuma conclusão experimental será registrada antes da execução e análise dos resultados.

# 1. Planejamento metodológico da otimização

Esta seção documenta as decisões tomadas antes da implementação do `RandomizedSearchCV`.

O planejamento considera os resultados dos experimentos anteriores, a capacidade computacional disponível, a necessidade de reprodutibilidade e o risco de overfitting.

O objetivo não é simplesmente encontrar a combinação de hiperparâmetros com a maior pontuação, mas identificar modelos que ofereçam um equilíbrio adequado entre desempenho preditivo, generalização e custo computacional.

## 1.1 — Análise e seleção dos modelos candidatos

### 1.1.1 — Contexto dos experimentos anteriores

O notebook `04_comparacao_features_aoi_courier.ipynb` avaliou oito experimentos, combinando dois algoritmos de classificação com quatro cenários de features.

Os algoritmos utilizados foram:

- Random Forest.
- XGBoost.

Os cenários investigaram a contribuição de informações operacionais, incluindo identificadores de áreas de interesse (`aoi_id`) e entregadores (`courier_id`).

A comparação considerou Average Precision, Recall@K, Precision@K, cardinalidade das variáveis categóricas, categorias inéditas, custo computacional e limitações metodológicas.

### 1.1.2 — Resultados selecionados

| Modelo | Cenário | Average Precision | Tempo de treinamento |
|---|---|---:|---:|
| Random Forest | D — Baseline + Courier | 0,226512 | 17,62 s |
| XGBoost | C — Baseline + AOI + Courier | 0,269095 | 7,80 s |
| XGBoost | D — Baseline + Courier | 0,268009 | 6,81 s |

Os tempos representam execuções individuais dos experimentos anteriores e não estimativas do tempo necessário para a otimização.

O XGBoost C apresentou a maior Average Precision entre os candidatos selecionados.

Entretanto, a diferença entre XGBoost C e D foi de apenas 0,001086.

Essa proximidade motivou a investigação de uma configuração com menor complexidade de features.

### 1.1.3 — Decisão metodológica

Foram selecionados para a primeira rodada de otimização:

**Random Forest D**

Permite investigar se a alteração dos hiperparâmetros melhora a capacidade de identificar coletas atrasadas.

**XGBoost D**

Apresentou desempenho próximo ao XGBoost C, com um conjunto de features mais simples e menor tempo de treinamento no experimento anterior.

**XGBoost C**

Será preservado como referência e poderá participar de uma segunda rodada de otimização, caso os resultados justifiquem o custo adicional.

A seleção não representa uma conclusão definitiva sobre qual algoritmo é superior.

## 1.2 — Definição dos hiperparâmetros e espaço de busca

### 1.2.1 — Conceito de hiperparâmetro

Hiperparâmetros são configurações definidas antes do treinamento e que controlam o comportamento do algoritmo.

Eles diferem dos parâmetros aprendidos durante o treinamento.

Por exemplo:

- O número de árvores de uma Random Forest é um hiperparâmetro.
- Os critérios de divisão aprendidos em cada árvore fazem parte do modelo ajustado.

A otimização busca combinações que possam melhorar a capacidade de generalização do classificador.

### 1.2.2 — Espaço de busca do Random Forest D

| Hiperparâmetro | Valores candidatos | Finalidade |
|---|---|---|
| `n_estimators` | 100, 200, 300 | Controlar a quantidade de árvores |
| `max_depth` | 10, 20, 30, None | Controlar a profundidade máxima |
| `min_samples_split` | 2, 5, 10 | Definir o mínimo de amostras para dividir um nó |
| `min_samples_leaf` | 1, 2, 5 | Definir o mínimo de amostras por folha |
| `max_features` | sqrt, 0.5, 1.0 | Controlar as features consideradas por divisão |

O espaço inicial contém 324 combinações possíveis.

O `RandomizedSearchCV` não avaliará todas essas combinações.

### 1.2.3 — Espaço de busca do XGBoost D

| Hiperparâmetro | Valores candidatos | Finalidade |
|---|---|---|
| `n_estimators` | 100, 200, 300 | Controlar o número de etapas de boosting |
| `max_depth` | 3, 5, 7 | Controlar a profundidade das árvores |
| `learning_rate` | 0.03, 0.05, 0.1 | Controlar a intensidade das atualizações |
| `subsample` | 0.7, 0.85, 1.0 | Controlar a amostragem de registros |
| `colsample_bytree` | 0.7, 0.85, 1.0 | Controlar a amostragem de features |
| `reg_lambda` | 1, 5, 10 | Controlar a regularização L2 |

O espaço inicial contém 729 combinações possíveis.

Os valores foram definidos para explorar diferentes níveis de complexidade e regularização, sem realizar uma busca exaustiva.

**Observação metodológica:** os espaços apresentados são propostas iniciais aprovadas para planejamento. Antes da execução, serão confrontados com as configurações originais dos estimadores e poderão receber ajustes justificados e documentados.

## 1.3 — Estratégia de validação e orçamento computacional

### 1.3.1 — Escolha do RandomizedSearchCV

O `RandomizedSearchCV`, disponibilizado pelo scikit-learn, permite avaliar um número limitado de combinações de hiperparâmetros selecionadas aleatoriamente.

Diferentemente do `GridSearchCV`, que percorre todas as combinações de uma grade, o `RandomizedSearchCV` permite controlar diretamente o orçamento de busca por meio do parâmetro `n_iter`.

Essa abordagem foi escolhida considerando o volume de dados e os recursos computacionais disponíveis.

### 1.3.2 — Configuração aprovada

| Configuração | Valor |
|---|---|
| Algoritmos | Random Forest D e XGBoost D |
| Método | RandomizedSearchCV |
| `n_iter` | 12 por modelo |
| Validação cruzada | 3 folds estratificados |
| Métrica principal | `average_precision` |
| `random_state` | 42 |
| Reajuste final | `refit=True`, previsto |
| Paralelismo | Controlado |
| Conjunto de teste | Preservado |

### 1.3.3 — Quantidade de treinamentos

Cada modelo terá 12 combinações de hiperparâmetros avaliadas em três folds.

Portanto:

**Random Forest D:** 12 × 3 = 36 ajustes de validação cruzada.

**XGBoost D:** 12 × 3 = 36 ajustes de validação cruzada.

**Total:** 72 ajustes de validação cruzada.

Com `refit=True`, serão realizados mais dois ajustes, um para cada modelo, utilizando os melhores hiperparâmetros encontrados e todo o conjunto de treinamento disponível para a busca.

Assim, o total previsto será de 74 ajustes, sem considerar eventuais execuções adicionais de diagnóstico.

### 1.3.4 — Validação cruzada estratificada

Será utilizado `StratifiedKFold`, com três divisões.

A estratificação procura preservar aproximadamente a proporção das classes em cada fold.

Essa característica é relevante porque o problema apresenta desbalanceamento entre coletas realizadas dentro da janela e coletas atrasadas.

A validação cruzada permite observar a estabilidade do desempenho em diferentes subconjuntos de treinamento.

Entretanto, a estratificação por linha não elimina possíveis dependências entre registros associados aos mesmos entregadores ou regiões. Essa limitação deverá ser considerada na interpretação dos resultados.

### 1.3.5 — Métrica principal

A métrica principal será Average Precision (AP).

A AP resume a relação entre precisão e revocação ao longo de diferentes limiares de classificação.

Essa métrica é apropriada para investigar a capacidade de priorização da classe positiva em problemas desbalanceados.

Também serão consideradas posteriormente métricas operacionais:

- Recall@K.
- Precision@K.
- Quantidade de alertas.
- Proporção de atrasos identificados dentro da capacidade operacional.

A maior Average Precision não garante automaticamente a melhor decisão operacional.

### 1.3.6 — Controle de overfitting

Serão observados:

- Desempenho médio entre os folds.
- Variação das métricas entre os folds.
- Diferença entre desempenho de treinamento e validação cruzada.
- Complexidade dos hiperparâmetros selecionados.
- Estabilidade dos resultados.
- Custo computacional.

Uma configuração que apresente desempenho elevado apenas nos dados de treinamento poderá indicar overfitting.

### 1.3.7 — Prevenção de data leakage

A otimização utilizará exclusivamente o conjunto de treinamento de 70%.

As transformações que aprendem informações a partir dos dados deverão ocorrer dentro do `Pipeline`, respeitando os limites de cada fold.

O conjunto de validação de 15%, utilizado anteriormente para comparar os cenários, permanecerá fora da busca de hiperparâmetros.

O conjunto de teste de 15% continuará reservado para a avaliação final.

Como a validação anterior já influenciou a escolha das features, essa dependência metodológica deverá ser reconhecida na interpretação dos resultados.

### 1.3.8 — Reprodutibilidade

Para permitir a reprodução dos experimentos, serão registrados:

- Versões das bibliotecas.
- Fonte e versão dos dados.
- Features utilizadas.
- Estratégia de particionamento.
- Configuração dos pipelines.
- Espaços de busca.
- Semente de aleatoriedade.
- Resultados da validação cruzada.
- Tempos de execução.
- Melhores hiperparâmetros encontrados.

O commit `6968e93` representa a referência versionada dos experimentos anteriores.

### 1.3.9 — Critérios para avançar

A otimização será iniciada somente após:

1. Confirmar as configurações originais dos modelos.
2. Reconstruir e validar o conjunto de treinamento.
3. Verificar as features e o pré-processamento.
4. Confirmar o espaço definitivo de hiperparâmetros.
5. Validar os recursos computacionais e o paralelismo.

O conjunto de teste não será utilizado durante essas verificações.

---

## 1.4 — Inspeção e reprodutibilidade dos experimentos

A próxima etapa iniciará a implementação técnica, verificando as configurações anteriores e os arquivos necessários para reconstruir os pipelines.

Ela seguirá a sequência didática definida para este laboratório: objetivo e metodologia, código Python comentado, explicação dos conceitos e, após a execução, resultados, interpretação e conclusão.

## Etapa 1.4 — Inspeção e reprodutibilidade dos experimentos

### Objetivo

Verificar as configurações utilizadas no notebook `04_comparacao_features_aoi_courier.ipynb`, que servirão de referência para a otimização de hiperparâmetros.

Nesta etapa, não serão treinados modelos nem carregados dados do conjunto de teste.

### Metodologia

A inspeção será realizada diretamente no arquivo `.ipynb` anterior, que utiliza o formato JSON.

Serão identificados:

1. As features utilizadas no cenário D (Baseline + Courier).
2. As configurações originais dos estimadores Random Forest e XGBoost.
3. A estrutura do pré-processamento e do Pipeline.
4. A existência das instruções de reconstrução do particionamento histórico.
5. A presença de uma fonte Silver persistida para reconstruir os dados.

O objetivo é confirmar as configurações de referência antes de implementá-las de forma independente no novo notebook.

### Critério de conclusão

A etapa será considerada concluída quando as configurações originais forem identificadas e a fonte de dados estiver localizada.

A reprodução dos dados e a validação do particionamento serão realizadas posteriormente, antes de qualquer treinamento.

In [1]:

# ============================================================
# LOGISTICS-05 — ETAPA 1.4
# Inspecao das configuracoes originais
# Nenhum treinamento sera executado
# ============================================================

import json
import ast
from pathlib import Path

# 1. Identificar a raiz do projeto.
raiz = Path.cwd().resolve()

if raiz.name == "notebooks":
    raiz = raiz.parent

# 2. Localizar o notebook anterior e a Silver.
notebook_anterior = (
    raiz / "notebooks"
    / "04_comparacao_features_aoi_courier.ipynb"
)

silver = (
    raiz / "data" / "silver"
    / "pickup_shanghai_silver.parquet"
)

print("Raiz:", raiz)
print("Notebook anterior:", notebook_anterior.is_file())
print("Silver:", silver.is_file())

assert notebook_anterior.is_file(), (
    "Notebook 04 nao encontrado."
)

assert silver.is_file(), (
    "Arquivo Silver nao encontrado."
)

# 3. Ler o JSON do notebook anterior.
with notebook_anterior.open(
    "r", encoding="utf-8"
) as arquivo:
    notebook = json.load(arquivo)

# 4. Reunir as celulas de codigo.
celulas_codigo = [
    "".join(celula.get("source", []))
    for celula in notebook.get("cells", [])
    if celula.get("cell_type") == "code"
]

# 5. Procurar definicoes relevantes.
termos = [
    "baseline =",
    "colunas_silver =",
    "train_test_split(",
    "cenarios =",
    "def montar_pipeline(",
    "features_d =",
]

for termo in termos:
    encontradas = [
        indice
        for indice, codigo in enumerate(
            celulas_codigo, start=1
        )
        if termo in codigo
    ]

    print(
        f"{termo:<25} "
        f"Celulas de codigo: {encontradas}"
    )

    assert encontradas, (
        f"Definicao nao encontrada: {termo}"
    )

# 6. Verificar a sintaxe Python das celulas.
for indice, codigo in enumerate(
    celulas_codigo, start=1
):
    ast.parse(codigo)

print("\nSintaxe Python das celulas: OK")
print("Total de celulas de codigo:", len(celulas_codigo))
print("Inspecao concluida sem executar treinamentos.")


Raiz: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform
Notebook anterior: True
Silver: True
baseline =                Celulas de codigo: [2]
colunas_silver =          Celulas de codigo: [2]
train_test_split(         Celulas de codigo: [3]
cenarios =                Celulas de codigo: [4]
def montar_pipeline(      Celulas de codigo: [4]
features_d =              Celulas de codigo: [8]

Sintaxe Python das celulas: OK
Total de celulas de codigo: 8
Inspecao concluida sem executar treinamentos.


### Etapa 1.4 — Explicação do código e conceitos Python

**1. Bibliotecas utilizadas**

- `json`: permite interpretar a estrutura de arquivos JSON, incluindo notebooks Jupyter.
- `ast`: significa *Abstract Syntax Tree* (árvore sintática abstrata). Permite analisar a sintaxe de código Python sem executá-lo.
- `pathlib.Path`: permite localizar e manipular caminhos de arquivos.

**2. Localização dos arquivos**

`Path.cwd()` identifica o diretório de trabalho atual.

`resolve()` obtém o caminho absoluto.

A condição `if raiz.name == "notebooks"` permite executar a inspeção tanto da raiz do projeto quanto da pasta `notebooks`.

**3. Validação com `assert`**

A instrução `assert` verifica uma condição.

Se a condição for falsa, o Python interrompe a execução com `AssertionError`.

Nesse caso, utilizamos `assert` para verificar se os arquivos existem e se as definições esperadas foram encontradas.

**4. Leitura do notebook**

`json.load()` converte o conteúdo do notebook em estruturas Python, como dicionários e listas.

A chave `cells` contém as células do notebook.

**5. List comprehension**

A expressão que cria `celulas_codigo` utiliza uma compreensão de lista (*list comprehension*).

Ela percorre as células, seleciona aquelas cujo tipo é `code` e reúne seu conteúdo.

**6. Busca das configurações**

O laço `for termo in termos` percorre as definições que queremos localizar.

A expressão `if termo in codigo` verifica se determinado texto está presente na célula.

Essa busca textual identifica a presença das definições, mas ainda não verifica seus valores ou sua correção semântica.

**7. Análise sintática com `ast.parse()`**

A função `ast.parse()` verifica se o código possui sintaxe Python válida, sem executá-lo.

Isso não comprova que o código funcionará durante a execução, nem que as variáveis e dependências estarão disponíveis.

**8. Segurança da inspeção**

O script apenas lê arquivos, analisa texto e verifica sintaxe.

Não executa as células do notebook anterior, não carrega registros da Silver e não treina modelos.

### Resultado esperado

A saída deverá apresentar a localização dos arquivos, as células em que as configurações foram encontradas e o resultado da análise sintática.

A interpretação definitiva será documentada após a execução.

### 1.4.1 — Resultados da inspeção

A inspeção do notebook `04_comparacao_features_aoi_courier.ipynb` foi executada com sucesso, sem realizar treinamentos ou modificar os arquivos originais.

**1. Verificação dos arquivos**

| Verificação | Resultado |
|---|---|
| Diretório raiz do projeto | Localizado |
| Notebook 04 | Encontrado |
| Arquivo Silver | Encontrado |
| Total de células de código | 8 |
| Verificação sintática | OK |
| Treinamentos executados | Nenhum |

**2. Localização das definições**

| Definição | Célula de código |
|---|---:|
| `baseline` | 2 |
| `colunas_silver` | 2 |
| `train_test_split` | 3 |
| `cenarios` | 4 |
| `montar_pipeline` | 4 |
| `features_d` | 8 |

Os números representam a posição das células de código, desconsiderando as células Markdown.

### 1.4.2 — Interpretação dos resultados

A inspeção confirma que:

1. O notebook anterior está disponível para consulta e reprodução das configurações.
2. A fonte Silver esperada está presente no projeto.
3. As definições procuradas foram encontradas no código.
4. As oito células de código passaram pela análise sintática com `ast.parse()`.
5. A inspeção não executou os modelos nem acessou o conjunto de teste.

Entretanto, essa verificação é estrutural.

Ela não confirma ainda os valores efetivos das features, dos hiperparâmetros ou das configurações do pipeline.

Também não comprova a integridade dos dados, a equivalência dos particionamentos ou a ausência de *data leakage*.

Essas verificações serão necessárias antes da otimização.

### 1.4.3 — Conclusão

**Status: inspeção estrutural concluída.**

Os arquivos e as definições necessárias foram localizados, permitindo avançar para a análise detalhada das configurações originais.

A próxima atividade será identificar os valores efetivos das features e dos hiperparâmetros, bem como verificar a estratégia de particionamento e o pré-processamento.

A otimização com `RandomizedSearchCV` permanecerá suspensa até a conclusão dessas validações.

### 1.4.4 — Inspeção detalhada das configurações originais

**Objetivo**

Examinar as definições identificadas na inspeção estrutural anterior, recuperando o código utilizado para configurar os dados, as features, os pipelines e os classificadores.

**Metodologia**

A leitura será realizada diretamente nas células de código do notebook anterior, sem executá-las.

Serão exibidas as células 2, 3, 4 e 8, identificadas na primeira inspeção.

Essa análise permitirá verificar os valores efetivos das configurações e avaliar sua reprodução no notebook atual.

**Critério de conclusão**

Identificar as configurações necessárias para reconstruir os experimentos Random Forest D e XGBoost D, preservando o particionamento e evitando vazamento de dados.

In [2]:

# ============================================================
# LOGISTICS-05 — ETAPA 1.4.4
# Inspecao detalhada das configuracoes originais
# Nenhum treinamento sera executado
# ============================================================

# Reutiliza a lista celulas_codigo criada na etapa 1.4.

celulas_inspecionar = [2, 3, 4, 8]

for numero in celulas_inspecionar:

    codigo = celulas_codigo[numero - 1]

    print("\n" + "=" * 75)
    print(f"CELULA DE CODIGO {numero}")
    print("=" * 75)

    for numero_linha, linha in enumerate(
        codigo.splitlines(), start=1
    ):
        print(f"{numero_linha:03d} | {linha}")

print("\nInspecao detalhada concluida.")



CELULA DE CODIGO 2
001 | baseline = ["region_id", "aoi_type", "lng", "lat", "janela_min", "aceite_ate_inicio_janela_min", "aceite_ate_fim_janela_min", "hora_aceite", "hora_inicio_janela", "hora_fim_janela"]
002 | colunas_silver = ["order_id", "accept_time", "time_window_start", "time_window_end", "coleta_apos_limite", "region_id", "aoi_type", "lng", "lat", "janela_min", "aceite_ate_inicio_janela_min", "aceite_ate_fim_janela_min", "aoi_id", "courier_id"]
003 | df = pd.read_parquet(silver, columns=colunas_silver)
004 | # Reproduz a máscara usada na etapa 2.7; validar também campos essenciais.
005 | assert df[["accept_time", "time_window_end", "coleta_apos_limite"]].notna().all().all(), "Revisar campos essenciais ausentes antes de prosseguir."
006 | df = df.loc[df["accept_time"] <= df["time_window_end"]].copy()
007 | df["hora_aceite"] = df["accept_time"].dt.hour
008 | df["hora_inicio_janela"] = df["time_window_start"].dt.hour
009 | df["hora_fim_janela"] = df["time_window_end"].dt.hour
01

### 1.4.5 — Explicação do código Python

**1. Reutilização de variáveis**

A variável `celulas_codigo` foi criada na inspeção anterior e contém o código-fonte das células do notebook `04`.

Sua reutilização evita a leitura desnecessária do arquivo JSON.

**2. Seleção das células**

A lista `celulas_inspecionar` contém os números das células que possuem as definições relevantes para a otimização.

**3. Índices de listas**

A expressão `celulas_codigo[numero - 1]` utiliza o índice da lista para acessar determinada célula.

Em Python, os índices começam em zero. Portanto, a célula de código 2 corresponde ao índice 1.

**4. Método `splitlines()`**

Divide uma string em linhas individuais, permitindo apresentar o código de maneira organizada.

**5. Função `enumerate()`**

Permite percorrer uma sequência e obter simultaneamente o número da posição e seu conteúdo.

O argumento `start=1` faz com que a numeração apresentada comece em 1.

**6. Formatação de strings**

A expressão `f"{numero_linha:03d}"` apresenta o número da linha com três dígitos, preenchendo com zeros à esquerda quando necessário.

**7. Segurança da execução**

O código apenas imprime o conteúdo das células selecionadas.

Não executa os scripts originais, não acessa os registros da Silver e não realiza treinamentos.

### 1.4.6 — Resultados, interpretação e conclusão da inspeção detalhada

#### 1. Resultados da inspeção

A análise das células 2, 3, 4 e 8 do notebook `04_comparacao_features_aoi_courier.ipynb` permitiu identificar as configurações utilizadas nos experimentos anteriores.

**1.1 — Features do cenário D**

O cenário D utiliza 11 features:

| Feature | Papel |
|---|---|
| `region_id` | Identificação da região |
| `aoi_type` | Tipo de área de interesse |
| `lng` | Longitude |
| `lat` | Latitude |
| `janela_min` | Duração da janela operacional |
| `aceite_ate_inicio_janela_min` | Intervalo entre aceite e início da janela |
| `aceite_ate_fim_janela_min` | Intervalo entre aceite e fim da janela |
| `hora_aceite` | Hora do aceite |
| `hora_inicio_janela` | Hora de início da janela |
| `hora_fim_janela` | Hora de término da janela |
| `courier_id` | Identificação do entregador |

A variável-alvo permanece `coleta_apos_limite`, que identifica coletas realizadas após o limite da janela operacional.

O cenário D não utiliza `aoi_id`.

**1.2 — Configurações originais dos classificadores**

| Hiperparâmetro | Random Forest D | XGBoost D |
|---|---|---|
| `n_estimators` | 100 | 200 |
| `max_depth` | 12 | 4 |
| `min_samples_leaf` | 20 | — |
| `max_features` | sqrt | — |
| `class_weight` | None | — |
| `learning_rate` | — | 0.05 |
| `subsample` | — | 0.8 |
| `colsample_bytree` | — | 0.8 |
| `objective` | — | binary:logistic |
| `eval_metric` | — | aucpr |
| `tree_method` | — | hist |
| `n_jobs` | 4 | 4 |
| `random_state` | 42 | 42 |

Os demais parâmetros não informados explicitamente utilizam os valores padrão das respectivas bibliotecas.

**1.3 — Pré-processamento**

O pipeline original utiliza:

- `ColumnTransformer` para separar variáveis categóricas e numéricas.
- `OneHotEncoder(handle_unknown="ignore", sparse_output=True)` para variáveis categóricas.
- `StandardScaler()` para variáveis numéricas.
- `sparse_threshold=1.0` para controlar a representação da saída.
- `Pipeline` para integrar pré-processamento e classificador.

No cenário D, as variáveis categóricas são `region_id`, `aoi_type` e `courier_id`.

As demais oito features são tratadas como numéricas.

**1.4 — Particionamento dos dados**

A população elegível é composta por 1.424.195 registros.

O particionamento original utiliza duas chamadas de `train_test_split`, com estratificação e `random_state=42`.

| Conjunto | Registros | Proporção aproximada |
|---|---:|---:|
| Treinamento | 996.936 | 70% |
| Validação | 213.629 | 15% |
| Teste reservado | 213.630 | 15% |

O notebook original verifica que os conjuntos não possuem índices em comum.

A otimização deverá preservar esse particionamento.

**1.5 — Critérios de elegibilidade**

Antes do particionamento, o código:

1. Verifica a ausência de nulos nos campos essenciais.
2. Mantém registros com `accept_time <= time_window_end`.
3. Calcula as três features de hora.
4. Confirma a população de 1.424.195 registros.
5. Verifica a ausência de nulos nas features do baseline.
6. Verifica que a variável-alvo contém as classes 0 e 1.
7. Confirma a presença de `aoi_id` e `courier_id`.

Não foi aplicado descarte de registros com base na raridade das categorias.

#### 2. Interpretação dos resultados

A inspeção permite reconstruir a configuração dos experimentos Random Forest D e XGBoost D no novo notebook.

Entretanto, foram identificados pontos que exigem atenção.

**2.1 — Espaço de busca versus configurações originais**

O planejamento inicial da otimização não contempla todos os valores utilizados anteriormente.

No Random Forest:

- `max_depth=12` não consta do espaço inicial.
- `min_samples_leaf=20` não consta do espaço inicial.

No XGBoost:

- `max_depth=4` não consta do espaço inicial.
- `subsample=0.8` não consta do espaço inicial.
- `colsample_bytree=0.8` não consta do espaço inicial.

Isso não invalida o planejamento, mas indica que devemos revisar o espaço de busca antes da execução.

A inclusão dos valores originais como candidatos não garante que a combinação original será sorteada pelo `RandomizedSearchCV`.

Por isso, a comparação com os modelos anteriores deverá permanecer explícita.

**2.2 — Reprodutibilidade**

A reprodução dos experimentos depende da preservação das features, dos critérios de elegibilidade, da ordenação dos registros, da estratégia de particionamento e das configurações de pré-processamento.

A utilização da mesma semente de aleatoriedade é necessária, mas não é suficiente, isoladamente, para garantir a reprodução dos conjuntos.

**2.3 — Prevenção de data leakage**

As features utilizadas devem representar informações disponíveis no momento em que o risco de atraso seria estimado.

O pré-processamento deverá ser ajustado exclusivamente nos dados de treinamento de cada fold.

A inspeção do código não constitui, por si só, uma validação completa da disponibilidade temporal das features.

#### 3. Conclusão

**Status: inspeção detalhada concluída.**

Foram identificadas as 11 features do cenário D, as configurações originais dos classificadores, o pipeline de pré-processamento, os critérios de elegibilidade e a estratégia de particionamento.

A próxima etapa será revisar o espaço de busca e reconstruir os dados e pipelines de forma independente no notebook `05`.

Nenhum treinamento foi realizado durante esta inspeção.

O conjunto de teste permanece reservado.

## 1.5 — Revisão e aprovação dos espaços de busca dos hiperparâmetros

### 1.5.1 — Objetivo

Revisar os espaços de busca definidos inicialmente para a otimização dos modelos Random Forest D e XGBoost D, considerando as configurações efetivamente utilizadas nos experimentos anteriores.

A revisão busca garantir que os valores originais dos hiperparâmetros estejam representados entre os candidatos, permitindo investigar configurações mais simples ou mais complexas.

A alteração será realizada sem aumentar o orçamento computacional previamente aprovado.

### 1.5.2 — Justificativa da revisão

Durante a inspeção detalhada do notebook `04_comparacao_features_aoi_courier.ipynb`, foram identificados valores de hiperparâmetros que não estavam contemplados nos espaços de busca inicialmente propostos.

**Random Forest D:**

- `max_depth=12`
- `min_samples_leaf=20`

**XGBoost D:**

- `max_depth=4`
- `subsample=0.8`
- `colsample_bytree=0.8`

A ausência desses valores não impediria a otimização, mas dificultaria a investigação de configurações próximas às utilizadas anteriormente.

Por esse motivo, decidiu-se incorporá-los aos espaços de busca.

### 1.5.3 — Espaço de busca aprovado: Random Forest D

| Hiperparâmetro | Valores aprovados |
|---|---|
| `n_estimators` | 100, 200, 300 |
| `max_depth` | 10, 12, 20, 30, None |
| `min_samples_split` | 2, 5, 10 |
| `min_samples_leaf` | 1, 2, 5, 20 |
| `max_features` | sqrt, 0.5, 1.0 |

**Quantidade de combinações possíveis:**

3 × 5 × 3 × 4 × 3 = **540 combinações**.

O espaço original possuía 324 combinações.

A ampliação permite explorar diferentes níveis de profundidade, complexidade e regularização das árvores.

### 1.5.4 — Espaço de busca aprovado: XGBoost D

| Hiperparâmetro | Valores aprovados |
|---|---|
| `n_estimators` | 100, 200, 300 |
| `max_depth` | 3, 4, 5, 7 |
| `learning_rate` | 0.03, 0.05, 0.1 |
| `subsample` | 0.7, 0.8, 0.85, 1.0 |
| `colsample_bytree` | 0.7, 0.8, 0.85, 1.0 |
| `reg_lambda` | 1, 5, 10 |

**Quantidade de combinações possíveis:**

3 × 4 × 3 × 4 × 4 × 3 = **1.728 combinações**.

O espaço original possuía 729 combinações.

A ampliação permite investigar configurações próximas às originais e alternativas com diferentes níveis de regularização e amostragem.

### 1.5.5 — Orçamento computacional

Apesar da ampliação dos espaços de busca, o orçamento de experimentação permanecerá inalterado.

| Configuração | Random Forest D | XGBoost D |
|---|---:|---:|
| Combinações possíveis | 540 | 1.728 |
| `n_iter` | 12 | 12 |
| Folds | 3 | 3 |
| Ajustes de validação cruzada | 36 | 36 |
| Reajustes finais previstos | 1 | 1 |

**Total previsto: 72 ajustes de validação cruzada e dois reajustes finais, considerando `refit=True`.**

A ampliação dos espaços não aumenta a quantidade de combinações avaliadas, mas pode alterar o tempo necessário para cada ajuste, dependendo dos hiperparâmetros sorteados.

### 1.5.6 — Limitações metodológicas

A utilização do `RandomizedSearchCV` com apenas 12 iterações representa uma exploração limitada dos espaços de busca.

Embora os valores originais tenham sido incluídos, não existe garantia de que suas combinações completas serão selecionadas aleatoriamente.

Portanto:

- Os resultados dos modelos originais permanecerão como referências independentes.
- A busca não será interpretada como uma identificação garantida do ótimo global.
- A comparação considerará Average Precision, estabilidade entre folds, custo computacional e métricas operacionais.
- Uma eventual segunda rodada de otimização deverá ser justificada pelos resultados obtidos.

Os demais parâmetros fixos dos classificadores serão preservados ou explicitamente documentados durante a reconstrução dos pipelines.

### 1.5.7 — Decisão metodológica

**Status: espaços de busca revisados e aprovados.**

Foram aprovadas as seguintes decisões:

1. Incorporar os valores originais dos hiperparâmetros aos espaços de busca.
2. Manter `n_iter=12` por algoritmo.
3. Utilizar três folds estratificados.
4. Manter `average_precision` como métrica principal.
5. Preservar o conjunto de teste de 15%, sem utilizá-lo durante a otimização.
6. Manter o pré-processamento dentro do Pipeline para evitar vazamento de informações entre os folds.
7. Avaliar os resultados antes de considerar uma segunda rodada de otimização.

### 1.5.8 — Conclusão

A revisão dos espaços de busca foi concluída e aprovada antes da implementação.

Os espaços definitivos contemplam as configurações originais dos modelos e alternativas de complexidade e regularização, mantendo o orçamento computacional estabelecido.

**Próxima etapa:** reconstrução e validação dos dados de treinamento e dos pipelines no notebook `05_otimizacao_hiperparametros_shanghai.ipynb`, antes da execução do `RandomizedSearchCV`.

Nenhum treinamento foi realizado nesta etapa.

## 2. Preparação e validação dos dados e pipelines

### 2.1 — Validação do ambiente e das dependências

#### Objetivo

Verificar se o ambiente Python possui as bibliotecas necessárias para reproduzir os experimentos anteriores e executar a otimização de hiperparâmetros.

#### Metodologia

Serão identificados:

- A versão e o executável do Python utilizado pelo notebook.
- As versões de Pandas, NumPy, SciPy, scikit-learn e XGBoost.
- A disponibilidade das classes necessárias para o pré-processamento, particionamento, treinamento e otimização.

A verificação será realizada sem carregar o dataset, criar pipelines ou treinar classificadores.

#### Critério de conclusão

Confirmar que as dependências estão disponíveis e que o notebook está utilizando o ambiente Python esperado para o projeto.

In [3]:

# ============================================================
# LOGISTICS-05 — ETAPA 2.1
# Validacao do ambiente e das dependencias
# Nenhum treinamento sera executado
# ============================================================

import sys
import importlib.metadata as metadata
from pathlib import Path

# 1. Identificar o interpretador Python.
print("Python:", sys.version.split()[0])
print("Executavel:", sys.executable)
print("Diretorio atual:", Path.cwd().resolve())

# 2. Consultar as versoes das dependencias instaladas.
dependencias = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "scikit-learn": "scikit-learn",
    "xgboost": "xgboost",
}

print("\nDependencias:")

for nome, pacote in dependencias.items():
    versao = metadata.version(pacote)
    print(f"{nome}: {versao}")

# 3. Verificar a disponibilidade das classes utilizadas.
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

print("\nImportacoes: OK")
print("Validacao do ambiente concluida.")


Python: 3.13.16
Executavel: c:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\.venv\Scripts\python.exe
Diretorio atual: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform\notebooks

Dependencias:
pandas: 2.3.3
numpy: 2.5.4
scipy: 1.18.1
scikit-learn: 1.9.1
xgboost: 3.4.1

Importacoes: OK
Validacao do ambiente concluida.


### 2.1.1 — Explicação do código Python

**1. Módulo `sys`**

O módulo `sys` fornece informações sobre o interpretador Python em execução.

- `sys.version`: identifica a versão do Python.
- `sys.executable`: apresenta o caminho do executável utilizado pelo kernel do notebook.

Essa verificação é importante porque o VS Code pode utilizar um interpretador diferente daquele configurado no terminal.

**2. Módulo `importlib.metadata`**

Permite consultar metadados dos pacotes instalados, incluindo suas versões.

A função `metadata.version()` identifica a versão instalada de uma distribuição Python.

**3. Dicionário Python**

A variável `dependencias` é um dicionário que associa o nome apresentado ao nome da distribuição instalada.

O método `.items()` permite percorrer simultaneamente as chaves e os valores.

**4. Importação das classes**

O código importa os componentes necessários às próximas etapas.

Entre eles:

- `train_test_split`: particionamento dos dados.
- `StratifiedKFold`: validação cruzada estratificada.
- `RandomizedSearchCV`: otimização de hiperparâmetros.
- `ColumnTransformer`: aplicação de transformações por grupo de colunas.
- `OneHotEncoder`: codificação de variáveis categóricas.
- `StandardScaler`: padronização de variáveis numéricas.
- `Pipeline`: integração entre pré-processamento e estimador.
- `RandomForestClassifier` e `XGBClassifier`: classificadores selecionados.
- `average_precision_score`: avaliação da métrica principal.

**5. Segurança da execução**

As importações verificam a disponibilidade das bibliotecas, mas não criam modelos treinados nem modificam os dados.

O conjunto de teste permanece preservado.

### Resultado esperado

A execução deverá apresentar a versão do Python, o caminho do interpretador, as versões das dependências e a confirmação das importações.

Os resultados efetivos serão documentados após a execução.

### 2.1.2 — Resultados, interpretação e conclusão

#### 1. Resultados da execução

A verificação do ambiente Python foi concluída sem erros.

**Interpretador Python**

| Item | Resultado |
|---|---|
| Versão do Python | 3.13.16 |
| Ambiente virtual | `.venv` do projeto |
| Diretório de execução | `Logistics_Intelligence_Platform/notebooks` |
| Importação das classes | OK |

**Dependências instaladas**

| Biblioteca | Versão identificada |
|---|---|
| Pandas | 2.3.3 |
| NumPy | 2.5.4 |
| SciPy | 1.18.1 |
| scikit-learn | 1.9.1 |
| XGBoost | 3.4.1 |

Todas as dependências consultadas foram encontradas.

As importações das classes necessárias para particionamento, validação cruzada, pré-processamento, classificação e avaliação foram realizadas com sucesso.

#### 2. Interpretação dos resultados

**2.1 — Ambiente virtual**

O caminho do executável confirma que o kernel do notebook está utilizando o Python instalado na `.venv` do projeto.

Isso reduz o risco de executar os experimentos com bibliotecas provenientes de outro ambiente Python.

**2.2 — Dependências**

As versões identificadas são compatíveis com as dependências utilizadas no planejamento atual.

A importação bem-sucedida das classes confirma sua disponibilidade no ambiente.

Entretanto, essa verificação não comprova que todos os pipelines executarão corretamente com os dados reais.

**2.3 — Reprodutibilidade**

O registro das versões contribui para a rastreabilidade dos experimentos.

Essas informações poderão ser utilizadas posteriormente na documentação do MLflow e na reprodução dos resultados.

**2.4 — Limitações da validação**

A inspeção não avaliou:

- Integridade e tipos das colunas do dataset.
- Reprodução dos conjuntos de treinamento e validação.
- Funcionamento efetivo dos pipelines.
- Consumo de memória.
- Tempo de treinamento.

Essas verificações serão realizadas nas próximas etapas.

#### 3. Conclusão

**Status: Etapa 2.1 concluída e validada.**

O ambiente Python está preparado para prosseguir com a reconstrução dos dados e pipelines dos classificadores Random Forest D e XGBoost D.

Nenhum modelo foi treinado e nenhum conjunto de dados foi carregado durante esta etapa.

O conjunto de teste permanece reservado.

**Próxima etapa: 2.2 — Reconstrução e validação da população elegível e do particionamento original.**

### 2.2 — Reconstrução e validação da população elegível e do particionamento

#### Objetivo

Reconstruir a população elegível utilizada nos experimentos anteriores e reproduzir o particionamento estratificado dos dados.

A reprodução é necessária para garantir que a otimização de hiperparâmetros utilize a mesma base de comparação dos modelos originais.

#### Metodologia

Serão realizadas as seguintes operações:

1. Carregar exclusivamente as colunas necessárias do arquivo Silver.
2. Verificar os campos essenciais e aplicar a regra de elegibilidade `accept_time <= time_window_end`.
3. Reconstruir as três features de hora utilizadas no baseline.
4. Validar a população esperada de 1.424.195 registros.
5. Reproduzir o particionamento estratificado com `random_state=42`.
6. Confirmar os tamanhos dos conjuntos e a ausência de índices compartilhados.

O conjunto de treinamento será preparado para as próximas etapas.

O conjunto de validação permanecerá separado da otimização.

O conjunto de teste terá somente seus índices reservados, sem criar um DataFrame de teste ou utilizar seus registros para avaliação.

#### Critério de conclusão

Confirmar que a população elegível, as features e os tamanhos dos conjuntos reproduzem as configurações do notebook anterior.

Nenhum modelo será treinado nesta etapa.

In [4]:

# ============================================================
# LOGISTICS-05 — ETAPA 2.2
# Reconstrucao da populacao e do particionamento original
# Nenhum modelo sera treinado
# ============================================================

import gc
import pandas as pd

# 1. Reproduzir exatamente as features do baseline.
baseline = [
    "region_id",
    "aoi_type",
    "lng",
    "lat",
    "janela_min",
    "aceite_ate_inicio_janela_min",
    "aceite_ate_fim_janela_min",
    "hora_aceite",
    "hora_inicio_janela",
    "hora_fim_janela",
]

features_d = baseline + ["courier_id"]

# 2. Reproduzir a selecao de colunas da Silver.
colunas_silver = [
    "order_id",
    "accept_time",
    "time_window_start",
    "time_window_end",
    "coleta_apos_limite",
    "region_id",
    "aoi_type",
    "lng",
    "lat",
    "janela_min",
    "aceite_ate_inicio_janela_min",
    "aceite_ate_fim_janela_min",
    "aoi_id",
    "courier_id",
]

# A variavel silver foi definida na Etapa 1.4.
assert silver.is_file(), "Arquivo Silver nao encontrado."

# 3. Carregar as colunas necessarias.
df = pd.read_parquet(silver, columns=colunas_silver)

# 4. Validar os campos essenciais.
campos_essenciais = [
    "accept_time",
    "time_window_end",
    "coleta_apos_limite",
]

assert df[campos_essenciais].notna().all().all(), (
    "Campos essenciais com valores ausentes."
)

# 5. Aplicar a mesma regra de elegibilidade.
df = df.loc[
    df["accept_time"] <= df["time_window_end"]
].copy()

# 6. Reconstruir as features temporais.
df["hora_aceite"] = df["accept_time"].dt.hour
df["hora_inicio_janela"] = (
    df["time_window_start"].dt.hour
)
df["hora_fim_janela"] = (
    df["time_window_end"].dt.hour
)

# 7. Validar a populacao e as features.
assert len(df) == 1_424_195, (
    "Populacao diferente do experimento original."
)

assert not df[baseline].isna().any().any(), (
    "Features do baseline apresentam valores ausentes."
)

assert not df[["aoi_id", "courier_id"]].isna().any().any(), (
    "Identificadores categoricos apresentam valores ausentes."
)

assert set(df["coleta_apos_limite"].unique()) == {0, 1}

assert len(features_d) == 11
assert len(features_d) == len(set(features_d))
assert set(features_d).issubset(df.columns)

print("Populacao elegivel:", len(df))
print("Total de atrasos:", int(df["coleta_apos_limite"].sum()))
print("Taxa de atrasos:", round(df["coleta_apos_limite"].mean(), 5))

# 8. Reproduzir o particionamento original.
indices = df.index
y = df["coleta_apos_limite"].astype(int)

idx_treino, idx_temporario = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

idx_validacao, idx_teste = train_test_split(
    idx_temporario,
    test_size=0.50,
    stratify=y.loc[idx_temporario],
    random_state=42,
)

# 9. Validar os tamanhos e a separacao dos indices.
assert len(idx_treino) == 996_936
assert len(idx_validacao) == 213_629
assert len(idx_teste) == 213_630

assert set(idx_treino).isdisjoint(idx_validacao)
assert set(idx_treino).isdisjoint(idx_teste)
assert set(idx_validacao).isdisjoint(idx_teste)

# 10. Materializar somente treino e validacao.
treino = df.loc[idx_treino].copy()
validacao = df.loc[idx_validacao].copy()

y_tr = treino["coleta_apos_limite"].astype(int)
y_va = validacao["coleta_apos_limite"].astype(int)

print("\nParticionamento:")
print("Treino:", len(treino))
print("Validacao:", len(validacao))
print("Teste reservado (indices):", len(idx_teste))

print("\nTaxas de atraso:")
print("Treino:", round(y_tr.mean(), 5))
print("Validacao:", round(y_va.mean(), 5))

# 11. Liberar estruturas intermediarias.
del df, y, indices, idx_temporario
gc.collect()

print("\nReconstrucao e validacoes concluidas.")


Populacao elegivel: 1424195
Total de atrasos: 36319
Taxa de atrasos: 0.0255

Particionamento:
Treino: 996936
Validacao: 213629
Teste reservado (indices): 213630

Taxas de atraso:
Treino: 0.0255
Validacao: 0.0255

Reconstrucao e validacoes concluidas.


### 2.2.3 — Explicação do código Python

**1. Reconstrução das features**

As listas `baseline` e `features_d` reproduzem as variáveis utilizadas no cenário D.

A expressão `baseline + ["courier_id"]` concatena duas listas Python, acrescentando a identificação do entregador às dez features originais.

**2. Leitura seletiva do Parquet**

A função `pd.read_parquet()` permite carregar um arquivo Parquet em um DataFrame Pandas.

O argumento `columns=colunas_silver` limita a leitura às colunas necessárias, reduzindo o consumo de memória em comparação com a leitura de todas as colunas.

**3. Validação com `assert`**

As instruções `assert` verificam condições que devem ser verdadeiras para reproduzir o experimento.

Se alguma condição não for atendida, a execução será interrompida para investigação.

**4. Filtragem com `.loc`**

A expressão `df.loc[condicao]` seleciona os registros que satisfazem uma condição lógica.

Neste caso, são mantidas as coletas cujo aceite ocorreu até o final da janela operacional.

O método `.copy()` cria uma cópia independente dos dados selecionados.

**5. Acessor `.dt`**

O Pandas disponibiliza o acessor `.dt` para operações sobre colunas temporais.

A propriedade `.dt.hour` extrai a hora de um valor datetime.

Ela é utilizada para reconstruir três variáveis derivadas dos horários operacionais.

**6. Particionamento estratificado**

A função `train_test_split()` divide os índices dos registros.

O parâmetro `stratify` procura preservar aproximadamente a proporção das classes nos conjuntos resultantes.

A primeira divisão separa 70% para treinamento e 30% para uma partição temporária.

A segunda divisão separa essa partição temporária em duas partes de aproximadamente 15% cada.

O parâmetro `random_state=42` permite reproduzir as divisões, desde que os dados e sua ordenação permaneçam equivalentes.

**7. Verificação de interseção entre conjuntos**

O método `.isdisjoint()` verifica se dois conjuntos não possuem elementos em comum.

Essa validação impede que um mesmo índice seja utilizado simultaneamente em treinamento, validação e teste.

**8. Gerenciamento de memória**

A instrução `del` remove referências a objetos intermediários.

A função `gc.collect()` solicita uma execução do coletor de lixo do Python.

Essas operações podem ajudar no gerenciamento de memória, mas não garantem a liberação imediata de toda a memória utilizada pelo processo.

**9. Preservação do conjunto de teste**

O código mantém somente `idx_teste`, sem materializar um DataFrame específico para esse conjunto.

Isso evita sua utilização acidental durante a otimização.

A validação da integridade dos índices não equivale a avaliar o desempenho do modelo no teste.

### Resultado esperado

A execução deverá confirmar a população elegível de 1.424.195 registros e reproduzir os tamanhos dos conjuntos originais.

As taxas de atraso serão verificadas após a execução.

Nenhum modelo será treinado nesta etapa.

### 2.2.4 — Resultados, interpretação e conclusão

#### 1. Resultados da execução

A reconstrução da população elegível e do particionamento original foi concluída sem erros.

**1.1 — População elegível**

| Indicador | Resultado |
|---|---:|
| Total de registros elegíveis | 1.424.195 |
| Coletas atrasadas | 36.319 |
| Coletas não atrasadas | 1.387.876 |
| Taxa de atraso | 2,55% |

Os resultados reproduzem a população utilizada nos experimentos anteriores.

**1.2 — Particionamento dos dados**

| Conjunto | Registros | Proporção aproximada |
|---|---:|---:|
| Treinamento | 996.936 | 70% |
| Validação | 213.629 | 15% |
| Teste reservado | 213.630 | 15% |
| **Total** | **1.424.195** | **100%** |

As verificações de tamanho e ausência de interseção entre os índices foram concluídas sem erros.

**1.3 — Distribuição da classe positiva**

| Conjunto | Taxa de atraso |
|---|---:|
| População elegível | 2,55% |
| Treinamento | 2,55% |
| Validação | 2,55% |

As taxas foram apresentadas com arredondamento de cinco casas decimais na execução.

O conjunto de teste permaneceu reservado, sem materialização de um DataFrame específico.

#### 2. Interpretação dos resultados

**2.1 — Reprodução dos experimentos anteriores**

A população elegível e os tamanhos dos conjuntos correspondem aos valores esperados do notebook `04`.

Isso indica que os critérios de elegibilidade e o procedimento de particionamento foram reproduzidos corretamente quanto às verificações realizadas.

A correspondência exata dos registros individuais com os experimentos anteriores depende também da preservação da fonte e da ordenação dos dados.

**2.2 — Desbalanceamento das classes**

Apenas 2,55% das coletas elegíveis apresentam atraso.

Isso significa que aproximadamente 97,45% pertencem à classe negativa.

Esse desbalanceamento reforça a importância de utilizar métricas apropriadas para avaliar a identificação de atrasos.

A acurácia isolada poderia apresentar valores elevados mesmo para um classificador com baixa capacidade de identificar coletas atrasadas.

Por esse motivo, manteremos Average Precision como métrica principal e Recall@K e Precision@K como métricas operacionais complementares.

**2.3 — Estratificação**

As taxas de atraso observadas no treinamento e na validação são consistentes com a população total.

Esse resultado é compatível com o particionamento estratificado utilizado.

A estratificação contribui para preservar a distribuição da variável-alvo entre os conjuntos, mas não elimina possíveis dependências entre registros associados aos mesmos entregadores ou regiões.

**2.4 — Prevenção de data leakage**

O conjunto de treinamento foi reconstruído separadamente da validação.

Os índices do conjunto de teste foram preservados sem sua utilização para treinamento ou avaliação.

A ausência de índices compartilhados reduz o risco de reutilização acidental dos mesmos registros entre os conjuntos.

Entretanto, essa verificação não elimina todas as formas possíveis de data leakage.

Na próxima etapa, o pré-processamento deverá permanecer dentro do Pipeline, garantindo que suas transformações sejam ajustadas exclusivamente com os dados de treinamento de cada fold.

#### 3. Conclusão

**Status: Etapa 2.2 concluída e validada.**

Foram confirmados:

- A população elegível de 1.424.195 registros.
- A presença de 36.319 coletas atrasadas.
- A taxa de atraso de 2,55%.
- A reprodução dos tamanhos dos conjuntos originais.
- A ausência de índices compartilhados entre treinamento, validação e teste.
- A preservação do conjunto de teste.

Nenhum modelo foi treinado durante esta etapa.

**Próxima etapa: 2.3 — Reconstrução e validação dos pipelines Random Forest D e XGBoost D.**

### 2.3 — Reconstrução e validação dos pipelines

**Objetivo:** reconstruir os pipelines Random Forest D e XGBoost D a partir das configurações originais do notebook `04_comparacao_features_aoi_courier.ipynb`.

**Procedimento:** reutilizar os conjuntos preparados na Etapa 2.2, configurar o pré-processamento e os classificadores e validar sua estrutura.

O pré-processamento será composto por:

- `OneHotEncoder` para variáveis categóricas.
- `StandardScaler` para variáveis numéricas.
- `ColumnTransformer` para combinar as transformações.
- `Pipeline` para integrar o pré-processamento ao classificador.

As configurações iniciais serão preservadas para garantir a rastreabilidade dos experimentos anteriores.

**Importante:** nenhum pipeline será ajustado com `.fit()` nesta etapa. Os parâmetros poderão ser alterados posteriormente pelo `RandomizedSearchCV`.

In [5]:

# ============================================================
# LOGISTICS-05 — ETAPA 2.3
# Reconstrucao e validacao dos pipelines
# Sem treinamento
# ============================================================

from sklearn.base import clone

# 1. Features do cenario D, definidas na Etapa 2.2.
categoricas = ["region_id", "aoi_type", "courier_id"]
numericas = [c for c in features_d if c not in categoricas]

# 2. Pre-processamento original.
preprocessamento = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categoricas,
        ),
        (
            "numericas",
            StandardScaler(),
            numericas,
        ),
    ],
    sparse_threshold=1.0,
)

# 3. Random Forest D — parametros originais.
pipeline_rf_d = Pipeline(
    steps=[
        ("preprocessamento", clone(preprocessamento)),
        (
            "classificador",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=12,
                min_samples_leaf=20,
                max_features="sqrt",
                class_weight=None,
                n_jobs=4,
                random_state=42,
            ),
        ),
    ]
)

# 4. XGBoost D — parametros originais.
pipeline_xgb_d = Pipeline(
    steps=[
        ("preprocessamento", clone(preprocessamento)),
        (
            "classificador",
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                n_jobs=4,
                random_state=42,
            ),
        ),
    ]
)

# 5. Validacoes estruturais.
assert len(features_d) == 11
assert len(categoricas) == 3
assert len(numericas) == 8

assert set(features_d) == set(categoricas + numericas)
assert set(features_d).issubset(treino.columns)
assert set(features_d).issubset(validacao.columns)

assert not treino[features_d].isna().any().any()
assert not validacao[features_d].isna().any().any()

assert list(pipeline_rf_d.named_steps) == [
    "preprocessamento",
    "classificador",
]

assert list(pipeline_xgb_d.named_steps) == [
    "preprocessamento",
    "classificador",
]

# Confirmar que os preprocessadores sao independentes.
assert (
    pipeline_rf_d.named_steps["preprocessamento"]
    is not pipeline_xgb_d.named_steps["preprocessamento"]
)

# Confirmar que nenhum pipeline foi ajustado.
assert not hasattr(
    pipeline_rf_d.named_steps["preprocessamento"],
    "transformers_",
)

assert not hasattr(
    pipeline_xgb_d.named_steps["preprocessamento"],
    "transformers_",
)

# 6. Resumo.
print("Cenario: D — Baseline + Courier")
print("Total de features:", len(features_d))
print("Categoricas:", categoricas)
print("Numericas:", numericas)

print("\nRandom Forest D:")
print(pipeline_rf_d.named_steps["classificador"])

print("\nXGBoost D:")
print(pipeline_xgb_d.named_steps["classificador"])

print("\nPipelines construidos e validados.")
print("Treinamento realizado: NAO")
print("Conjunto de teste utilizado: NAO")


Cenario: D — Baseline + Courier
Total de features: 11
Categoricas: ['region_id', 'aoi_type', 'courier_id']
Numericas: ['lng', 'lat', 'janela_min', 'aceite_ate_inicio_janela_min', 'aceite_ate_fim_janela_min', 'hora_aceite', 'hora_inicio_janela', 'hora_fim_janela']

Random Forest D:
RandomForestClassifier(max_depth=12, min_samples_leaf=20, n_jobs=4,
                       random_state=42)

XGBoost D:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_we

### 2.3.3 — Explicação do código

A célula reconstrói os pipelines dos dois classificadores, preservando as configurações utilizadas nos experimentos anteriores.

**`ColumnTransformer`:** permite aplicar transformações diferentes a grupos de colunas. As variáveis categóricas são codificadas pelo `OneHotEncoder`, enquanto as numéricas passam pelo `StandardScaler`.

**`Pipeline`:** encadeia o pré-processamento e o classificador. Durante a validação cruzada, essa estrutura permite ajustar as transformações exclusivamente com os dados de treinamento de cada fold, reduzindo o risco de data leakage.

**`clone()`:** cria uma nova instância não ajustada do pré-processador para cada pipeline. Assim, Random Forest e XGBoost não compartilham o mesmo objeto de transformação.

**`named_steps`:** permite acessar os componentes do Pipeline pelo nome, facilitando a inspeção de configurações e, posteriormente, a otimização de hiperparâmetros.

**`assert`:** verifica a quantidade e a disponibilidade das features, a ausência de valores nulos e a estrutura dos pipelines.

**Parâmetros originais:** os classificadores foram configurados com os valores utilizados no notebook anterior. Esses valores constituem referências para a nova busca, mas não representam necessariamente os melhores hiperparâmetros.

**Limitação:** a validação realizada é estrutural. Ela não garante que os modelos executarão sem erros durante o treinamento nem avalia desempenho ou consumo de recursos.

Nenhum treinamento ou ajuste de pré-processamento será realizado nesta etapa.

### 2.3.4 — Resultados e conclusão

A reconstrução dos pipelines Random Forest D e XGBoost D foi concluída com sucesso.

Foram confirmadas as 11 features do cenário D, sendo três categóricas e oito numéricas.

Os dois pipelines foram construídos com suas configurações originais, incluindo os componentes de pré-processamento e os classificadores.

As verificações estruturais foram concluídas sem erros, e os pré-processadores foram instanciados de forma independente.

**Interpretação:** os pipelines estão preparados para a próxima etapa de otimização. Os modelos originais já foram treinados e avaliados no notebook anterior, mas os novos objetos ainda não foram ajustados.

A validação estrutural não mede desempenho, tempo de treinamento ou consumo de memória. Esses aspectos serão observados durante os novos experimentos.

**Conclusão:** Etapa 2.3 concluída. Nenhum treinamento foi realizado e o conjunto de teste permaneceu reservado.

**Próxima etapa:** 2.4 — Definição da estrutura de persistência dos resultados dos experimentos.


### Etapa 2.4.12 — Configuração reutilizável do MLflow Tracking

**Objetivo**

Estabelecer uma conexão persistente com o MLflow Tracking para registrar e recuperar os futuros experimentos de otimização de hiperparâmetros dos classificadores Random Forest D e XGBoost D.

**Arquitetura adotada**

- MLflow: versão 3.18.0.
- Backend Store: SQLite (`mlruns/mlflow.db`).
- Artifact Store: `mlruns/artifacts/`.
- Experimento: `NSLAB_Logistics_Shanghai_Hyperparameter_Optimization`.
- Identificação das execuções: `run_id`.

**Procedimento**

1. Identificar a raiz do repositório.
2. Construir o endereço do banco SQLite.
3. Configurar o Tracking URI.
4. Recuperar o experimento existente.
5. Validar sua identificação e localização dos artefatos.

**Critérios de validação**

- Conexão com o SQLite existente.
- Experimento identificado pelo ID `1`.
- Artifact Store correspondente ao diretório configurado.
- Nenhuma nova execução criada.

Esta etapa não realiza treinamento, otimização ou avaliação de modelos. O conjunto de teste permanece preservado.


In [1]:

# ============================================================
# LOGISTICS-05 — ETAPA 2.4.12
# Configuração reutilizável do MLflow Tracking
# ============================================================

from pathlib import Path
import mlflow

# 1. Identificar a raiz do repositório
diretorio_atual = Path.cwd().resolve()

if (diretorio_atual / "requirements.txt").is_file():
    raiz_projeto = diretorio_atual
elif (diretorio_atual.parent / "requirements.txt").is_file():
    raiz_projeto = diretorio_atual.parent
else:
    raise FileNotFoundError(
        "Raiz do projeto não identificada."
    )

# 2. Definir os caminhos persistentes
caminho_banco = raiz_projeto / "mlruns" / "mlflow.db"
caminho_artefatos = raiz_projeto / "mlruns" / "artifacts"

# 3. Confirmar que o banco já existe
if not caminho_banco.is_file():
    raise FileNotFoundError(
        f"Banco MLflow não encontrado: {caminho_banco}"
    )

# 4. Configurar o Tracking URI
tracking_uri = f"sqlite:///{caminho_banco.as_posix()}"
mlflow.set_tracking_uri(tracking_uri)

# 5. Recuperar o experimento existente
nome_experimento = (
    "NSLAB_Logistics_Shanghai_Hyperparameter_Optimization"
)

experimento = mlflow.get_experiment_by_name(nome_experimento)

if experimento is None:
    raise RuntimeError(
        "Experimento MLflow não encontrado."
    )

# 6. Validar a configuração persistida
artifact_uri_esperada = caminho_artefatos.as_uri()

assert experimento.experiment_id == "1"
assert experimento.lifecycle_stage == "active"
assert experimento.artifact_location == artifact_uri_esperada

# 7. Disponibilizar o identificador para etapas posteriores
experiment_id = experimento.experiment_id

# 8. Apresentar os resultados
print("CONFIGURAÇÃO MLFLOW TRACKING")
print("-" * 50)
print("Versão MLflow:", mlflow.__version__)
print("Raiz do projeto:", raiz_projeto)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment ID:", experiment_id)
print("Experimento:", experimento.name)
print("Artifact URI:", experimento.artifact_location)
print("Status:", experimento.lifecycle_stage)
print("-" * 50)
print("Configuração validada com sucesso.")


CONFIGURAÇÃO MLFLOW TRACKING
--------------------------------------------------
Versão MLflow: 3.18.0
Raiz do projeto: C:\Users\nslab\Documents\labs\Logistics_Intelligence_Platform
Tracking URI: sqlite:///C:/Users/nslab/Documents/labs/Logistics_Intelligence_Platform/mlruns/mlflow.db
Experiment ID: 1
Experimento: NSLAB_Logistics_Shanghai_Hyperparameter_Optimization
Artifact URI: file:///C:/Users/nslab/Documents/labs/Logistics_Intelligence_Platform/mlruns/artifacts
Status: active
--------------------------------------------------
Configuração validada com sucesso.



### Explicação do código — MLflow Tracking

**1. `Path.cwd().resolve()`**

Identifica o diretório de execução do notebook e obtém seu caminho absoluto.

Como o notebook pode ser executado a partir da raiz do repositório ou da pasta `notebooks/`, o código identifica a raiz utilizando a presença do arquivo `requirements.txt`.

**2. Backend Store**

O arquivo `mlruns/mlflow.db` contém os registros persistentes dos experimentos.

O SQLite permite consultar parâmetros, métricas, identificadores e informações das execuções sem repetir os treinamentos.

**3. `mlflow.set_tracking_uri()`**

Configura o endereço utilizado pelo MLflow para acessar os registros dos experimentos.

A configuração é realizada diretamente no notebook, sem depender de variáveis definidas anteriormente no terminal PowerShell.

**4. `mlflow.get_experiment_by_name()`**

Recupera um experimento previamente registrado.

Diferentemente de `mlflow.create_experiment()`, esse método não cria um novo experimento.

**5. `experiment_id`**

Identifica o experimento que agrupará as execuções dos classificadores Random Forest D e XGBoost D.

Cada execução terá seu próprio `run_id`, permitindo distinguir os resultados de diferentes configurações de hiperparâmetros.

**6. Artifact Store**

O diretório `mlruns/artifacts/` armazenará os arquivos associados às execuções, como resultados e modelos serializados.

O código verifica se o endereço corresponde ao que foi configurado anteriormente.

**7. `assert`**

Verifica automaticamente condições necessárias para continuar o laboratório.

Caso alguma condição não seja atendida, a execução é interrompida, evitando utilizar um experimento ou armazenamento diferente do planejado.

**Resultado esperado**

Ao concluir a execução, o notebook estará conectado ao MLflow Tracking e preparado para registrar os futuros experimentos de otimização.

Nenhum modelo será treinado nesta etapa.



### Etapa 2.4.12 — Resultados, interpretação e conclusão

#### 1. Resultados observados

A configuração reutilizável do MLflow Tracking foi executada e validada com sucesso no notebook `05_otimizacao_hiperparametros_shanghai.ipynb`.

Foram confirmadas as seguintes propriedades:

| Propriedade | Resultado |
|---|---|
| Versão do MLflow | 3.18.0 |
| Backend Store | SQLite |
| Banco de dados | `mlruns/mlflow.db` |
| Experiment ID | 1 |
| Nome do experimento | `NSLAB_Logistics_Shanghai_Hyperparameter_Optimization` |
| Artifact Store | `mlruns/artifacts/` |
| Lifecycle Stage | active |

Todas as verificações automáticas foram concluídas sem erros.

#### 2. Interpretação dos resultados

O notebook conseguiu identificar a raiz do repositório e estabelecer uma conexão com o banco SQLite previamente inicializado.

O experimento existente foi recuperado pelo nome, mantendo seu identificador original (`experiment_id = 1`).

A localização dos artefatos também correspondeu à configuração persistida anteriormente.

Esses resultados demonstram que o notebook pode reutilizar a infraestrutura do MLflow Tracking sem depender da sessão PowerShell utilizada na configuração inicial.

A persistência dos registros já havia sido validada nas etapas anteriores, incluindo a recuperação de parâmetros, métricas e artefatos por meio do `run_id`.

#### 3. Conclusão

A configuração reutilizável do MLflow Tracking foi concluída.

O ambiente está preparado para registrar os futuros experimentos de otimização dos classificadores Random Forest D e XGBoost D.

A infraestrutura adotada permite:

- Registrar parâmetros e métricas dos experimentos.
- Associar resultados a identificadores únicos (`run_id`).
- Persistir informações no SQLite.
- Armazenar artefatos localmente.
- Consultar e comparar execuções pela interface gráfica do MLflow.
- Integrar posteriormente os resultados aos arquivos Parquet.

Nenhum treinamento ou procedimento de otimização foi executado nesta etapa.

O conjunto de teste permanece reservado para a avaliação final dos modelos.
